## Importação das Bibliotecas

In [114]:
import pandas as pd
import requests 
import sqlalchemy
import psycopg2
import json
import numpy as np
import re
from datetime import datetime

In [115]:
pd.set_option('display.max_columns', None)

In [116]:
pd.set_option('display.max_colwidth', None)

## Extração do Staging do Verdana

In [117]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Conectar ao banco PostgreSQL usando SQLAlchemy
url = URL.create(
    drivername="postgresql+psycopg2",
    username="admin_pg",
    password="1nbr@ndsIB",
    host="192.168.50.102",
    port=5432,
    database="db_bi_ti"
)
# Criar a engine de conexão
engine = create_engine(url)
# Ler a tabela
df = pd.read_sql('SELECT * FROM f_chamados_verdanadesk', engine)

## Campo de Tratamento dos dados

In [118]:
df.columns = df.columns.str.strip()
df.columns

Index(['id', 'requerente', 'titulo', 'status', 'tipo', 'origem_abertura',
       'prioridade', 'categoria', 'categoria_completa', 'localizacao',
       'nome_tecnico', 'grupo', 'tempo_chamado', 'data_criacao',
       'ultima_atualizacao', 'tempo_atribuicao', 'tempo_solucao',
       'data_solucao', 'data_fechamento', 'expirado', 'tempo_resposta'],
      dtype='str')

Tratamento da Localização

In [119]:
map_filial = {
    
}

In [120]:
map_localidade = {

    'MORUMBI': ('SÃO PAULO', 'SP'),

    'IBIRAPUERA': ('SÃO PAULO', 'SP'),

    'HIGIENOPOLIS': ('SÃO PAULO', 'SP'),

    'OSCAR FREIRE': ('SÃO PAULO', 'SP'),

    'LEBLON': ('RIO DE JANEIRO', 'RJ'),

    'BARRA SHOPPING': ('RIO DE JANEIRO', 'RJ'),

    'RIO SUL': ('RIO DE JANEIRO', 'RJ'),

    'PATIO SAVASSI': ('BELO HORIZONTE', 'MG'),

    'IGUATEMI FORTALEZA': ('FORTALEZA', 'CE'),

    'SALVADOR SHOPPING': ('SALVADOR', 'BA'),

    'MIDWAY NATAL': ('NATAL', 'RN'),

    'BALNEARIO CAMBORIU': ('BALNEÁRIO CAMBORIÚ', 'SC')
}

In [121]:
map_marca = {

    'VR': 'VR',
    'TH': 'TOMMY HILFIGER',
    'ELLUS': 'ELLUS',
    'RC': "RICHARD'S",
    'SL': 'SALINAS',
    'BS': 'BRANDS HOUSE'
}

In [122]:
map_cidade = {

    'MORUMBI': 'SÃO PAULO',
    'IBIRAPUERA': 'SÃO PAULO',
    'HIGIENOPOLIS': 'SÃO PAULO',
    'LEBLON': 'RIO DE JANEIRO',
    'BARRA SHOPPING': 'RIO DE JANEIRO',
    'PATIO SAVASSI': 'BELO HORIZONTE',
    'IGUATEMI FORTALEZA': 'FORTALEZA'
}

In [123]:
map_estado = {

    'SÃO PAULO': 'SP',
    'RIO DE JANEIRO': 'RJ',
    'BELO HORIZONTE': 'MG',
    'FORTALEZA': 'CE'
}

In [125]:
map_regiao = {

    'SP': 'SUDESTE',
    'RJ': 'SUDESTE',
    'MG': 'SUDESTE',
    'SC': 'SUL',
    'BA': 'NORDESTE',
    'CE': 'NORDESTE',
    'RN': 'NORDESTE'
}

In [126]:
def limpar_unidade(valor):

    if pd.isnull(valor):
        return np.nan

    valor = re.sub(
        r'^[A-Z]{2,10}[0-9]{0,3}\\s*-\\s*',
        '',
        valor
    )

    return valor.strip()

In [127]:
def classificar_tipo_operacao(valor):

    if pd.isnull(valor):
        return np.nan

    valor = valor.upper()

    if 'OUTLET' in valor:
        return 'OUTLET'

    elif valor.startswith('CD'):
        return 'CENTRO DISTRIBUIÇÃO'

    elif 'HUB' in valor:
        return 'HUB LOGÍSTICO'

    elif 'CORPORATIVO' in valor:
        return 'CORPORATIVO'

    else:
        return 'LOJA'

In [128]:
def extrair_marca(valor):

    if pd.isnull(valor):
        return np.nan

    valor = valor.upper()

    if valor.startswith('VR'):
        return 'VR'

    elif valor.startswith('TH'):
        return 'TOMMY HILFIGER'

    elif valor.startswith('ELLUS'):
        return 'ELLUS'

    elif valor.startswith('RC'):
        return "RICHARD'S"

    elif valor.startswith('SL'):
        return 'SALINAS'

    elif valor.startswith('BS'):
        return 'BRANDS HOUSE'

    elif valor.startswith('HUB'):
        return 'HUB'

    elif valor.startswith('CD'):
        return 'CENTRO DISTRIBUIÇÃO'

    elif valor.startswith('INBRANDS'):
        return 'INBRANDS'

    else:
        return 'NÃO IDENTIFICADO'

In [129]:
valores_invalidos = [
    'NÃO ATRIBUÍDO',
    'NAO ATRIBUIDO',
    'NULL',
    'NONE',
    ''
]

Tratamento de Data

In [132]:
inicio = df['data_criacao'].min()
fim = pd.Timestamp.today()

In [133]:
map_dias = {
    'Monday': 'SEGUNDA',
    'Tuesday': 'TERÇA',
    'Wednesday': 'QUARTA',
    'Thursday': 'QUINTA',
    'Friday': 'SEXTA',
    'Saturday': 'SÁBADO',
    'Sunday': 'DOMINGO'
}

In [134]:
campos_data = [
    'data_criacao',
    'tempo_atribuicao',
    'tempo_resposta',
    'data_solucao',
    'data_fechamento'
]

In [139]:
calendario = pd.date_range(
    start=inicio,
    end=fim,
    freq='D'
)

In [140]:
for col in campos_data:

    df[col] = pd.to_datetime(
        df[col],
        errors='coerce'
    )

In [141]:
df['data_abertura_merge'] = (
    df['data_criacao']
    .dt.date
)

df['data_fechamento_merge'] = (
    df['data_fechamento']
    .dt.date
)

Tratamento dos dados de Grupos

In [142]:
campos_texto = [
    'status',
    'grupo',
    'prioridade',
    'nome_tecnico'
]

for col in campos_texto:

    df[col] = (
        df[col]
        .astype(str)
        .str.upper()
        .str.strip()
    )

In [143]:

valores_nao_atribuidos = [
    'NÃO ATRIBUÍDO',
    'NAO ATRIBUIDO',
    'NULL',
    'NONE',
    ''
]

for col in campos_texto:

    df[col] = np.where(
        df[col].isin(valores_nao_atribuidos),
        np.nan,
        df[col]
    )

In [144]:
### Tratamento dos Dados de Grupo, fazendo a separação:

# Coluna Area_responsavel
map_area = {
    'TI | BUSINESS INTELLIGENCE N1' : 'TI',
    'TI | E-COMMERCE' : 'TI',
    'TI | GOVERNANÇA' : 'TI',
    'TI | INFRAESTRUTURA' : 'TI',
    'TI | LINE CARTUCHOS' : 'TI',
    'TI | SISTEMAS' : 'TI',
    'TI | SISTEMAS N2' : 'TI',
    'TI | SUPORTE A LOJAS (RETAGUARDA)' : 'TI',
    'TI | SUPORTE A LOJAS N2' : 'TI',
    'TI | SUPORTE LOCAL': 'TI',
    'TI | SUPORTE A LOJAS': 'TI',
    'TI | PROJETOS': 'TI',
    'TI | '
    'ADMINISTRAÇÃO DE PESSOAL': 'RH',
    'REMUNERAÇÃO' : 'RH',
    'DEPARTAMENTO FISCAL' : 'FISCAL',
    'MANUTENÇÃO | HUB' : 'MANUTENÇÃO',
    'MANUTENÇÃO | LOJAS' : 'MANUTENÇÃO',
    'GPP / SEGURANÇA (HUB)' : 'TI',
    'GPP / SEGURANÇA (LOJAS)' : 'TI',
    'CROWN IT - ATUALIZAÇÃO DE VERSÃO LINX' : 'TI',
    'CROWN IT N2' : 'TI'

}

#Coluna Torre_responsavel
map_torre = {
    'TI | SUPORTE LOCAL': 'INFRAESTRUTURA',
    'TI | SUPORTE A LOJAS': 'SUPORTE LOJAS',
    'TI | SUPORTE A LOJAS (RETAGUARDA)' : 'SUPORTE A LOJAS',
    'TI | SUPORTE A LOJAS N2' : 'SUPORTE A LOJAS',
    'TI | SISTEMAS': 'SISTEMAS',
    'TI | SISTEMAS N2': 'SISTEMAS',
    'TI | PROJETOS': 'PROJETOS',
    'TI | E-COMMERCE' : 'E-COMMERCE',
    'TI | GOVERNANÇA' : 'GOVERNANÇA',
    'TI | INFRAESTRUTURA' : 'INFRAESTRUTURA',
    'TI | BUSINESS INTELLIGENCE N1' : 'BUSINESS INTELLIGENCE',
    'TI | LINE CARTUCHOS' : 'SUPORTE LOCAL',
    'TI | '
    'ADMINISTRAÇÃO DE PESSOAL': 'RH OPERAÇÕES',
    'REMUNERAÇÃO' : 'RH OPERAÇÕES',
    'CROWN IT - ATUALIZAÇÃO DE VERSÃO LINX' : 'SISTEMAS',
    'CROWN IT N2' : 'SISTEMAS',
    'DEPARTAMENTO FISCAL' : 'FISCAL',
    'MANUTENÇÃO HUB': 'MANUTENÇÃO',
    'MANUTENÇÃO LOJAS': 'MANUTENÇÃO',
    'GPP / SEGURANÇA (HUB)': 'INFRAESTRUTURA',
    'GPP / SEGURANÇA (LOJAS)': 'INFRAESTRUTURA'
}

In [145]:
# Criando os campos email_requerente e dominio_email
df = df.rename(columns={'requerente': 'email_requerente'})
df['dominio_email'] = df['email_requerente'].str.split('@').str[1]
df['dominio_email'] = df['dominio_email'].fillna('Não atribuído')


In [146]:
df['localizacao'] = (
    df['localizacao']
    .astype(str)
    .str.upper()
    .str.strip()
)

df['codigo_unidade'] = (
    df['localizacao']
    .map(map_filial)
)


In [147]:
# Tratamento do campo nome tecnico e grupo

df['nome_tecnico'] = df['nome_tecnico'].astype(str)
df['grupo'] = df['grupo'].astype(str)


df[['departamento', 'grupo_descricao']] = (
    df['grupo']
    .str.split('|', n=1, expand=True)
)

df['nome_tecnico'] = df['nome_tecnico'].str.strip()
df['departamento'] = df['departamento'].str.strip()
df['grupo_descricao'] = df['grupo_descricao'].str.strip()


In [148]:
df = df.replace(pd.NaT, "")

# Tratamento do campo Categoria
df['categoria'] = df['categoria'].astype(str)

df[['categoria', 'subcategoria']] = (
    df['categoria_completa']
    .str.split('>', n=1, expand=True)
)

# Remover espaços extras e dados nulos
df['categoria'] = df['categoria'].str.strip()
df['subcategoria'] = df['subcategoria'].str.strip().fillna('Não atribuído')


Tratamento de Status

In [149]:
# Criando a coluna de Status_macro

df['status'] = (df['status'].astype(str)
                .str.upper()
                .str.strip()
                )

map_status_macro = {'NOVO': 'Aberto',
                    'EM ATENDIMENTO (ATRIBUÍDO)': 'Em atendimento', 
                    'EM ATENDIMENTO (PLANEJADO)': 'Em atendimento',
                    'FECHADO': 'Fechado',
                    'SOLUCIONADO': 'Fechado',
                    'PENDENTE': 'Pendente',
                    }

df['status_macro'] = (df['status'].map(map_status_macro))


## Métricas

In [150]:
### Tempo Primeira Resposta
df['tempo_primeira_resposta_horas'] = (
    (
        df['tempo_resposta']
        - df['data_criacao']
    )
    .dt.total_seconds()
    / 3600
)

In [151]:
### Tempo Resolução
df['tempo_resolucao_horas'] = (
    (
        df['data_solucao']
        - df['data_criacao']
    )
    .dt.total_seconds()
    / 3600
)

## SLA (Regra Corporativa)

In [152]:
condicoes = [

    df['data_solucao'].notnull(),

    (
        df['data_solucao'].isnull()
        &
        df['status'].isin([
            'ABERTO',
            'EM ANDAMENTO'
        ])
    ),

    (
        df['data_solucao'].isnull()
        &
        df['status'].str.contains(
            'PENDENTE',
            na=False
        )
    ),

    (
        df['nome_tecnico'].isnull()
    )
]

In [153]:
valores = [
    'FINALIZADO',
    'EM ABERTO',
    'PENDENTE',
    'NÃO ATRIBUÍDO'
]

In [154]:
df['status_sla'] = np.select(
    condicoes,
    valores,
    default='NÃO CLASSIFICADO'
)

In [155]:
map_sla = {
    'CRÍTICA': 2,
    'ALTA': 4,
    'MÉDIA': 8,
    'BAIXA': 24
}

In [156]:
df['sla_previsto_horas'] = (
    df['prioridade']
    .map(map_sla)
)

In [157]:
df['flag_sla_dentro_prazo'] = np.where(

    df['tempo_resolucao_horas'].isnull(),

    np.nan,

    np.where(
        df['tempo_resolucao_horas']
        <=
        df['sla_previsto_horas'],
        1,
        0
    )
)

In [158]:
def classificar_faixa_sla(tempo):

    if pd.isnull(tempo):
        return 'EM ANDAMENTO'

    elif tempo <= 1:
        return 'ATÉ 1H'

    elif tempo <= 4:
        return '1H A 4H'

    elif tempo <= 8:
        return '4H A 8H'

    elif tempo <= 18:
        return '8H A 18H'

    else:
        return 'ACIMA 18H'

In [159]:
df['faixa_sla'] = (
    df['tempo_resolucao_horas']
    .apply(classificar_faixa_sla)
)

## Criação das Tabelas de Dimensão

In [191]:
### 5.1 D_DATA
D_DATA = pd.DataFrame({
    'data_completa': calendario
})

D_DATA['data_merge'] = (
    D_DATA['data_completa']
    .dt.date
)

D_DATA['data_sk'] = (
    D_DATA['data_completa']
    .dt.strftime('%Y%m%d')
    .astype(int)
)

### Colunas Temporais ###

D_DATA['ano'] = (
    D_DATA['data_completa']
    .dt.year
)

D_DATA['mes'] = (
    D_DATA['data_completa']
    .dt.month
)

D_DATA['nome_mes'] = (
    D_DATA['data_completa']
    .dt.month_name()
)

D_DATA['trimestre'] = (
    D_DATA['data_completa']
    .dt.quarter
)

D_DATA['dia'] = (
    D_DATA['data_completa']
    .dt.day
)

D_DATA['dia_semana_num'] = (
    D_DATA['data_completa']
    .dt.dayofweek
)

D_DATA['dia_semana'] = (
    D_DATA['data_completa']
    .dt.day_name()
)

D_DATA['dia_semana'] = (
    D_DATA['dia_semana']
    .map(map_dias)
)

D_DATA['eh_dia_util'] = np.where(
    D_DATA['dia_semana_num'] < 5,
    1,
    0
)


In [192]:
### 5.2 D_REQUERENTE
D_REQUERENTE = (
        df[['email_requerente', 'dominio_email']]
        .drop_duplicates()
        .sort_values(by='email_requerente')
        .reset_index(drop=True)
)
D_REQUERENTE['requerente_sk'] = D_REQUERENTE.index + 1
D_REQUERENTE

,email_requerente,dominio_email,requerente_sk
0,,Não atribuído,1
1,Breno Maciel,Não atribuído,2
2,Felipe Gonçalves,Não atribuído,3
3,Gabriele Bispo,Não atribuído,4
4,Jefferson Barbosa,Não atribuído,5
...,...,...,...
970,yellen.moreira@inbrands.com.br,inbrands.com.br,971
971,ygor.parecy@inbrands.com.br,inbrands.com.br,972
972,yngrid.ferreira@inbrands.com.br,inbrands.com.br,973
973,yuki.sato@inbrands.com.br,inbrands.com.br,974


In [193]:
### 5.3 D_TECNICO
D_TECNICO = (
    df[
        df['nome_tecnico'].notnull()
    ][['nome_tecnico', 'departamento']]
    .drop_duplicates()
    .sort_values(by='nome_tecnico')
    .reset_index(drop=True)
)

# Criação da flag_sem_tecnico
df['flag_sem_tecnico'] = np.where(
    df['nome_tecnico'].isnull(),
    1,
    0
)

# Validar quantidade sem técnico
if df['flag_sem_tecnico'].sum() > 0:
    print("Sem Técnico")

df_sem_tecnico = df[
    df['flag_sem_tecnico'] == 1
]
D_TECNICO['tecnico_sk'] = D_TECNICO.index + 1
D_TECNICO

,nome_tecnico,departamento,tecnico_sk
0,,TI,1
1,,MANUTENÇÃO,2
2,,GPP / SEGURANÇA (LOJAS),3
3,,ADMINISTRAÇÃO DE PESSOAL,4
4,,CROWN IT - ATUALIZAÇÃO DE VERSÃO LINX,5
...,...,...,...
64,VALDIR BOSCO DA SILVA JUNIOR,TI,65
65,VANUSA PEREIRA SALES,ADMINISTRAÇÃO DE PESSOAL,66
66,WESLLEY DE SANTANA MOREIRA,TI,67
67,WESLLEY DE SANTANA MOREIRA,CROWN IT N2,68


In [194]:
D_LOCALIZACAO = (
    df[['localizacao']]
    .drop_duplicates()
    .rename(
        columns={
            'localizacao': 'localizacao_original'
        }
    )
    .reset_index(drop=True)
)
D_LOCALIZACAO['localizacao_original'] = (
    D_LOCALIZACAO['localizacao_original']
    .astype(str)
    .str.upper()
    .str.strip()
)

D_LOCALIZACAO['localizacao_original'] = np.where(
    D_LOCALIZACAO['localizacao_original']
    .isin(valores_invalidos),
    np.nan,
    D_LOCALIZACAO['localizacao_original']
)

D_LOCALIZACAO['unidade_padronizada'] = (
    D_LOCALIZACAO['localizacao_original']
    .str.replace(
        r'^[A-Z]{2,15}[0-9]{0,3}\\s*-\\s*',
        '',
        regex=True
    )
    .str.strip()
)

D_LOCALIZACAO['marca'] = (
    D_LOCALIZACAO['localizacao_original']
    .str.extract(r'^([A-Z]{2,10})')
    [0]
    .map(map_marca)
    .fillna('INBRANDS')
)

D_LOCALIZACAO['tipo_operacao'] = np.select(

    [
        D_LOCALIZACAO['localizacao_original']
        .str.contains('OUTLET', na=False),
        D_LOCALIZACAO['localizacao_original']
        .str.contains('HUB', na=False),
        D_LOCALIZACAO['localizacao_original']
        .str.startswith('CD', na=False),
        D_LOCALIZACAO['localizacao_original']
        .str.contains('CORPORATIVO', na=False)
    ],

    [
        'OUTLET',
        'HUB',
        'CENTRO DISTRIBUIÇÃO',
        'CORPORATIVO'
    ],
    default='LOJA'
)

D_LOCALIZACAO['cidade'] = (
    D_LOCALIZACAO['unidade_padronizada']
    .map(
        lambda x:
        map_localidade.get(x, (None, None))[0]
    )
)

D_LOCALIZACAO['uf'] = (
    D_LOCALIZACAO['unidade_padronizada']
    .map(
        lambda x:
        map_localidade.get(x, (None, None))[1]
    )
)

D_LOCALIZACAO['regiao'] = (
    D_LOCALIZACAO['uf']
    .map(map_regiao)
)

D_LOCALIZACAO['endereco_padronizado'] = (
    D_LOCALIZACAO['unidade_padronizada']
    + ' - '
    + D_LOCALIZACAO['cidade']
    .fillna('NÃO IDENTIFICADO')
    + ' - '
    + D_LOCALIZACAO['uf']
    .fillna('NI')
)

D_LOCALIZACAO['codigo_filial'] = (
    D_LOCALIZACAO['marca']
    .str[:3]
    +
    '_'
    +
    D_LOCALIZACAO.index.astype(str)
)

D_LOCALIZACAO['cnpj_filial'] = np.nan
D_LOCALIZACAO['flag_outlet'] = np.where(
    D_LOCALIZACAO['tipo_operacao'] == 'OUTLET',
    1,
    0
)

D_LOCALIZACAO['flag_hub'] = np.where(
    D_LOCALIZACAO['tipo_operacao'] == 'HUB',
    1,
    0
)

D_LOCALIZACAO['flag_cd'] = np.where(
    D_LOCALIZACAO['tipo_operacao'] == 'CENTRO DISTRIBUIÇÃO',
    1,
    0
)

D_LOCALIZACAO = (
    D_LOCALIZACAO
    .reset_index(drop=True)
)

D_LOCALIZACAO['localizacao_sk'] = (
    D_LOCALIZACAO.index + 1
)

D_LOCALIZACAO['data_cadastro_dw'] = (
    datetime.now()
)

D_LOCALIZACAO = D_LOCALIZACAO[

    [
        'localizacao_sk',
        'codigo_filial',
        'cnpj_filial',
        'localizacao_original',
        'unidade_padronizada',
        'marca',
        'cidade',
        'uf',
        'regiao',
        'endereco_padronizado',
        'tipo_operacao',
        'flag_outlet',
        'flag_hub',
        'flag_cd',
        'data_cadastro_dw'
    ]
]

In [ ]:
# # 5.4 D_LOCALIZACAO
# D_LOCALIZACAO = (
#     df[['localizacao']]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )

# D_LOCALIZACAO['localizacao'] = (

#     D_LOCALIZACAO['localizacao']
#     .astype(str)
#     .str.upper()
#     .str.strip()
# )

# D_LOCALIZACAO['localizacao'] = np.where(

#     D_LOCALIZACAO['localizacao'].isin(
#         valores_invalidos
#     ),

#     np.nan,

#     D_LOCALIZACAO['localizacao']
# )

# D_LOCALIZACAO['marca'] = (
#     D_LOCALIZACAO['localizacao']
#     .apply(extrair_marca)
# )

# D_LOCALIZACAO['tipo_operacao'] = (

#     D_LOCALIZACAO['localizacao']
#     .apply(classificar_tipo_operacao)
# )

# D_LOCALIZACAO['unidade_padronizada'] = (

#     D_LOCALIZACAO['localizacao']
#     .apply(limpar_unidade)
# )

# D_LOCALIZACAO['cidade'] = (

#     D_LOCALIZACAO['unidade_padronizada']
#     .map(map_cidade)
# )

# D_LOCALIZACAO['estado'] = (

#     D_LOCALIZACAO['cidade']
#     .map(map_estado)
# )

# D_LOCALIZACAO['regiao'] = (

#     D_LOCALIZACAO['estado']
#     .map(map_regiao)
# )

# D_LOCALIZACAO['flag_outlet'] = np.where(

#     D_LOCALIZACAO['localizacao']
#     .str.contains(
#         'OUTLET',
#         na=False
#     ),

#     1,

#     0
# )

# D_LOCALIZACAO['flag_hub'] = np.where(

#     D_LOCALIZACAO['localizacao']
#     .str.contains(
#         'HUB',
#         na=False
#     ),

#     1,

#     0
# )

# D_LOCALIZACAO['flag_cd'] = np.where(

#     D_LOCALIZACAO['localizacao']
#     .str.startswith(
#         'CD',
#         na=False
#     ),

#     1,

#     0
# )

# D_LOCALIZACAO['flag_corporativo'] = np.where(

#     D_LOCALIZACAO['localizacao']
#     .str.contains(
#         'CORPORATIVO',
#         na=False
#     ),

#     1,

#     0
# )

# D_LOCALIZACAO = (

#     D_LOCALIZACAO
#     .sort_values(
#         by='unidade_padronizada'
#     )
#     .reset_index(drop=True)
# )

# D_LOCALIZACAO['localizacao_sk'] = (

#     D_LOCALIZACAO.index + 1
# )

In [195]:
### 5.5 D_CATEGORIA
D_CATEGORIA = (
    df[['categoria', 'subcategoria']]
    .drop_duplicates()
    .sort_values(by='categoria')
    .reset_index(drop=True)
)
D_CATEGORIA['categoria_sk'] = D_CATEGORIA.index + 1
D_CATEGORIA

,categoria,subcategoria,categoria_sk
0,Abertura Via Chat Indevida,Não atribuído,1
1,Acessos,Liberação De Acesso Linx,2
2,Acessos,Banco De Dados,3
3,Acessos,Liberação De Pastas Da Rede,4
4,Acessos,Site,5
...,...,...,...
514,Verdanadesk,Inclusão De Usuário,515
515,Verdanadesk,Ajustes,516
516,Verdanadesk,Perfil,517
517,Verdanadesk,Criação De Categoria,518


In [ ]:
### 5.6 D_GRUPO
D_GRUPO = (
    df[['grupo']]
    .drop_duplicates()
)

D_GRUPO['grupo'] = (
    D_GRUPO['grupo']
    .astype(str)
    .str.upper()
    .str.strip()
)

D_GRUPO['area_responsavel'] = (
    D_GRUPO['grupo']
    .map(map_area)
)

D_GRUPO['torre_servico'] = (
    D_GRUPO['grupo']
    .map(map_torre)
)

D_GRUPO = (
    D_GRUPO
    .sort_values(by='grupo')
    .reset_index(drop=True)
)

D_GRUPO['grupo_sk'] = (
    D_GRUPO.index + 1
)

In [197]:
### 5.7 D_STATUS
D_STATUS = (
    df[['status', 'status_macro']]
    .drop_duplicates()
    .sort_values(by='status')
    .reset_index(drop=True)
)
D_STATUS['status_sk'] = D_STATUS.index + 1
D_STATUS

,status,status_macro,status_sk
0,EM ATENDIMENTO (ATRIBUÍDO),Em atendimento,1
1,EM ATENDIMENTO (PLANEJADO),Em atendimento,2
2,FECHADO,Fechado,3
3,NOVO,Aberto,4
4,PENDENTE,Pendente,5
5,SOLUCIONADO,Fechado,6


In [198]:
### 5.8 D_TIPO
D_TIPO = (
    df[['tipo']]
    .drop_duplicates()
    .sort_values(by='tipo')
    .reset_index(drop=True)
)
D_TIPO['tipo_sk'] = D_TIPO.index + 1
D_TIPO

,tipo,tipo_sk
0,Incidente,1
1,Requisição,2


In [199]:
### 5.9 D_ORIGEM_ABERTURA
D_ORIGEM_ABERTURA = (
    df[['origem_abertura']]
    .drop_duplicates()
    .sort_values(by='origem_abertura')
    .reset_index(drop=True)
)
D_ORIGEM_ABERTURA['origem_abertura_sk'] = D_ORIGEM_ABERTURA.index + 1
D_ORIGEM_ABERTURA

,origem_abertura,origem_abertura_sk
0,Chat,1
1,Direct,2
2,E-Mail,3
3,Formcreator,4
4,Helpdesk,5
5,Other,6


In [200]:
### 5.10 D_PRIORIDADE
D_PRIORIDADE = (
    df[['prioridade']]
    .drop_duplicates()
    .sort_values(by='prioridade')
    .reset_index(drop=True)
)
D_PRIORIDADE['prioridade_sk'] = D_PRIORIDADE.index + 1
D_PRIORIDADE

,prioridade,prioridade_sk
0,ALTA,1
1,BAIXA,2
2,CRÍTICA,3
3,MUITO ALTA,4
4,MUITO BAIXA,5
5,MÉDIA,6


In [ ]:
### 5.11 D_SLA
D_SLA = (
    df[['sla_previsto_horas','status_sla','faixa_sla']]
    .drop_duplicates()
    .sort_values(
        by=['sla_previsto_horas']
    )
    .reset_index(drop=True)
)
D_SLA['sla_sk'] = (
    D_SLA.index + 1
)

## Criação da Tabela Fato 

In [ ]:
F_CHAMADO = df.merge(
    D_DATA[['data_sk','data_merge']],
    left_on='data_abertura_merge',
    right_on='data_merge',
    how='left'
)

F_CHAMADO = F_CHAMADO.rename(
    columns={'data_sk': 'data_abertura_sk'}
)

In [ ]:
F_CHAMADO = F_CHAMADO.merge(
    D_DATA[['data_sk','data_merge']],
    left_on='data_fechamento_merge',
    right_on='data_merge',
    how='left'
)

In [ ]:
F_CHAMADO = F_CHAMADO.rename(
    columns={'data_sk': 'data_fechamento_sk'}
)

In [208]:
# Criação da Tabela fato

F_CHAMADOS = df.merge(
    D_TECNICO,
    how='left',
    left_on=['nome_tecnico', 'departamento'],
    right_on=['nome_tecnico', 'departamento']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_CATEGORIA,
    how='left',
    left_on=['categoria', 'subcategoria'],
    right_on=['categoria', 'subcategoria']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_STATUS,
    how='left',
    left_on=['status','status_macro'],
    right_on=['status','status_macro']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_REQUERENTE,
    how='left',
    left_on=['email_requerente', 'dominio_email'],
    right_on=['email_requerente', 'dominio_email']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_TIPO,
    how='left',
    left_on=['tipo'],
    right_on=['tipo']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_ORIGEM_ABERTURA,
    how='left',
    left_on=['origem_abertura'],
    right_on=['origem_abertura']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_GRUPO,
    how='left',
    left_on=['grupo'],
    right_on=['grupo']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_SLA[['sla_sk','sla_previsto_horas',
         'status_sla','faixa_sla']],
    on=['sla_previsto_horas','status_sla','faixa_sla'],
    how='left'
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_DATA[['data_sk','data_merge']],
    left_on='data_fechamento_merge',
    right_on='data_merge',
    how='left'
)

F_CHAMADO['localizacao'] = (
    F_CHAMADO['localizacao']
    .astype(str)
    .str.upper()
    .str.strip()
)

F_CHAMADO = F_CHAMADO.merge(
    D_LOCALIZACAO[['localizacao_sk','localizacao_original']],
    left_on='localizacao',
    right_on='localizacao_original',
    how='left'
)

F_CHAMADOS = F_CHAMADOS.drop(columns=
                             ['nome_tecnico', 
                              'grupo',
                              'departamento', 
                              'categoria', 
                              'categoria', 
                              'subcategoria',   
                              'status',
                              'email_requerente',
                              'tipo',
                              'origem_abertura',
                              'prioridade',
                              'localizacao',
                              'categoria_completa',
                              'tempo_chamado',
                              'data_criacao',
                              'ultima_atualizacao',
                              'tempo_atribuicao',
                              'tempo_solucao',
                              'data_fechamento',
                              'data_solucao',
                              'expirado',
                              'tempo_resposta',
                              'grupo_descricao',
                              'dominio_email',
                              'status_macro',
                              'sla_previsto_horas',
                              'tempo_primeira_resposta_horas',
                              'codigo_unidade',
                              'tempo_resolucao_horas',
                              'status_sla',
                              'faixa_sla',
                              'data_abertura_merge',
                              'data_fechamento_merge',
                              'area_responsavel',
                              'torre_servico',
                              'data_merge',
                              'data_sk']
                              )
F_CHAMADOS = F_CHAMADOS.rename(columns={'id': 'id_chamado'})
F_CHAMADOS['chamado_sk'] = F_CHAMADOS.index + 1
F_CHAMADOS

,id_chamado,titulo,flag_sla_dentro_prazo,flag_sem_tecnico,tecnico_sk,categoria_sk,status_sk,requerente_sk,tipo_sk,origem_abertura_sk,grupo_sk,sla_sk,chamado_sk
0,27782,Liberação De Acesso A Inclusão/Alteração Das Telas,NaN,0,1,2,4,963,2,4,18,11,1
1,27781,Tela De Faturamento Lento,NaN,0,1,337,4,286,1,4,18,11,2
2,27780,Linx Lento,NaN,0,1,306,4,628,1,4,18,23,3
3,27779,Studio Photos: Solicitação Para Abrir Todas As Saídas,NaN,0,2,34,1,110,1,4,8,2,4
4,27778,Linx Lento,NaN,0,1,306,4,187,1,4,18,23,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27189,71,Notas Fiscais Não Aparecem Para Entrada.,1.0,0,62,225,3,821,2,4,20,18,27190
27190,70,Nota Fiscal De Doação,1.0,0,62,225,3,821,2,4,20,16,27191
27191,69,Peça Não Cadastrada,0.0,0,62,363,3,822,2,4,20,25,27192
27192,68,Estou Precisando Passar Venda No Linxpos E A Tela De Vendas Não Está Abrindo. Parou De Funcionar Hoje A Tarde Depois De Uma Atualização Feira Por Aí Mesmo.,0.0,0,35,325,3,392,1,1,20,9,27193


In [206]:
df.columns = df.columns.str.strip()
df.columns

Index(['id', 'email_requerente', 'titulo', 'status', 'tipo', 'origem_abertura',
       'prioridade', 'categoria', 'categoria_completa', 'localizacao',
       'nome_tecnico', 'grupo', 'tempo_chamado', 'data_criacao',
       'ultima_atualizacao', 'tempo_atribuicao', 'tempo_solucao',
       'data_solucao', 'data_fechamento', 'expirado', 'tempo_resposta',
       'data_abertura_merge', 'data_fechamento_merge', 'dominio_email',
       'codigo_unidade', 'departamento', 'grupo_descricao', 'subcategoria',
       'status_macro', 'tempo_primeira_resposta_horas',
       'tempo_resolucao_horas', 'status_sla', 'sla_previsto_horas',
       'flag_sla_dentro_prazo', 'faixa_sla', 'flag_sem_tecnico'],
      dtype='str')

## Exportação via Excel

In [207]:
# Criar arquivos excel

with pd.ExcelWriter("Tratamento_chamados.xlsx") as writer:
    D_DATA.to_excel(writer, sheet_name="D_DATA", index=False)
    D_REQUERENTE.to_excel(writer, sheet_name="D_REQUERENTE", index=False)
    D_TECNICO.to_excel(writer, sheet_name="D_TECNICO", index=False)
    D_LOCALIZACAO.to_excel(writer, sheet_name="D_LOCALIZACAO", index=False)
    D_CATEGORIA.to_excel(writer, sheet_name="D_CATEGORIA", index=False)
    D_GRUPO.to_excel(writer, sheet_name="D_GRUPO", index=False)
    D_STATUS.to_excel(writer, sheet_name="D_STATUS", index=False)
    D_TIPO.to_excel(writer, sheet_name="D_TIPO", index=False)
    D_ORIGEM_ABERTURA.to_excel(writer, sheet_name="D_ORIGEM_ABERTURA", index=False)
    D_PRIORIDADE.to_excel(writer, sheet_name="D_PRIORIDADE", index=False)
    D_SLA.to_excel(writer, sheet_name="D_SLA", index=False)
    F_CHAMADOS.to_excel(writer, sheet_name="F_CHAMADOS", index=False)